# TPCx-AI UC8: Trip Type Classification with Distributed XGBoost

Multi-class classification on retail order data using the TPCx-AI benchmark (unofficial).

**Pipeline:** Delta Tables → PySpark SQL (join + aggregate) → SparkXGBClassifier

## Prerequisites

- A cluster running **Databricks Runtime for Machine Learning** (ships with `xgboost.spark`).
- Workers: `Standard_DS4_v2` (28 GB, 8 vCPU).
- **Spark config applied in the published benchmark** (set on the cluster under Advanced options → Spark config):
  - `spark.task.cpus = 8` — each XGBoost worker uses all 8 vCPUs on the node
  - `spark.executor.memory = 10g` — reduces the JVM heap reservation from the ~18 GB default, leaving headroom for XGBoost's native allocator
  - `num_workers` matches the worker-node count (one XGBoost worker per node; computed in the config cell as `NUM_NODES - 1`)
- **Cluster size is fixed at creation time** — run one scale factor at a time and resize the cluster between runs.

  The driver coordinates orchestration but does no training compute, so total nodes = `workers + 1` (except for single-node clusters, where the driver also trains).

  Recommended sizing per scale factor:

  - SF=1: single-node cluster (1 total node)
  - SF=10: 2 workers (3 total nodes)
  - SF=100: 4 workers (5 total nodes)
  - SF=1000: 24 workers (25 total nodes)

  SF=1000 OOMs with 12 workers — start with ≥24. Set `SCALE_FACTORS` to a single value (e.g. `[100]`), run, resize the cluster, change `SCALE_FACTORS`, run again. Results from each run append to `RESULT_TABLE`.
- TPCx-AI UC8 data loaded into Delta tables `TPCXAI_V2_SF{N}.{ORDERS, LINEITEM, PRODUCT}` — one schema per scale factor, matching the TPCx-AI kit's output layout. See `setup_dbx.ipynb` for the load script (CSVs from S3 → Hive metastore Delta tables); generate the raw CSVs first with the official [TPCx-AI kit](https://www.tpc.org/tpcx-ai/).

## Running

1. Attach this notebook to your cluster.
2. In the config cell, adjust `SCALE_FACTORS`, `RESULT_TABLE`, and `CONFIG`.
3. Run all cells. Per-run timings are printed inline and appended to `RESULT_TABLE`.

## Pipeline Components

- `spark.table()` — reads from Delta tables (loaded via `setup_dbx.ipynb`)
- PySpark SQL — three-way join + GROUP BY for feature engineering (weekday one-hot + per-department quantity sums)
- `dense_rank()` — encodes trip-type labels to 0-based indices
- `VectorAssembler` — packs feature columns into a single vector column
- `SparkXGBClassifier` — distributed XGBoost training across Spark executors

In [ ]:
import timeit, re
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from xgboost.spark import SparkXGBClassifier

In [ ]:
print(f"Driver: {spark.conf.get('spark.driver.host')}")
print(f"Workers: {spark.conf.get('spark.databricks.clusterUsageTags.clusterWorkers')}")
print(f"spark.task.cpus: {spark.sparkContext.getConf().get('spark.task.cpus', 'not set')}")


In [ ]:
print("spark.executor.memory:", spark.conf.get("spark.executor.memory", "not set"))
print("spark.executor.memoryOverhead:", spark.conf.get("spark.executor.memoryOverhead", "not set"))
print("spark.executor.pyspark.memory:", spark.conf.get("spark.executor.pyspark.memory", "not set"))

# Per-executor JVM heap usage
mem_status = spark.sparkContext._jsc.sc().getExecutorMemoryStatus()
it = mem_status.iterator()
while it.hasNext():
    entry = it.next()
    print(entry.toString())

In [ ]:
# ---- Config ----
SCALE_FACTORS = [1000]                     # set ONE scale factor per run; resize the cluster between runs (see prerequisites). Tables must exist as TPCXAI_V2_SF{N}
NUM_RUNS      = 5                          # repeat each scale factor for variance reporting

# Where benchmark timings are appended (one row per phase per run).
# TODO: change to a database/schema you can write to.
RESULT_TABLE  = "YOUR_DB.UC8_RESULTS"

# Free-form label for this benchmark configuration (e.g. cluster type, tuning preset).
# Stored alongside each result row so multiple configs can be compared in the summary.
CONFIG        = "tuned_8cpu"

DBR_VERSION   = spark.conf.get("spark.databricks.clusterUsageTags.sparkVersion", "unknown")
NUM_NODES     = int(spark.conf.get("spark.databricks.clusterUsageTags.clusterWorkers", "0")) + 1  # +1 for driver
NUM_WORKERS   = max(NUM_NODES - 1, 1)   # one XGBoost worker per non-driver node (clamped to >= 1 for single-node clusters); configure spark.task.cpus = cores-per-node so each worker uses the whole node

WEEKDAYS = ["MONDAY", "TUESDAY", "WEDNESDAY", "THURSDAY", "FRIDAY", "SATURDAY", "SUNDAY"]

# 68 product departments from the TPCx-AI UC8 spec; one summed-quantity feature per department.
DEPARTMENTS = [
    "FINANCIAL SERVICES", "SHOES", "PERSONAL CARE", "PAINT AND ACCESSORIES", "DSD GROCERY",
    "MEAT - FRESH & FROZEN", "DAIRY", "PETS AND SUPPLIES", "HOUSEHOLD CHEMICALS/SUPP",
    "IMPULSE MERCHANDISE", "PRODUCE", "CANDY, TOBACCO, COOKIES", "GROCERY DRY GOODS",
    "BOYS WEAR", "FABRICS AND CRAFTS", "JEWELRY AND SUNGLASSES", "MENS WEAR", "ACCESSORIES",
    "HOME MANAGEMENT", "FROZEN FOODS", "SERVICE DELI", "INFANT CONSUMABLE HARDLINES",
    "PRE PACKED DELI", "COOK AND DINE", "PHARMACY OTC", "LADIESWEAR", "COMM BREAD", "BAKERY",
    "HOUSEHOLD PAPER GOODS", "CELEBRATION", "HARDWARE", "BEAUTY", "AUTOMOTIVE",
    "BOOKS AND MAGAZINES", "SEAFOOD", "OFFICE SUPPLIES", "LAWN AND GARDEN", "SHEER HOSIERY",
    "WIRELESS", "BEDDING", "BATH AND SHOWER", "HORTICULTURE AND ACCESS", "HOME DECOR", "TOYS",
    "INFANT APPAREL", "LADIES SOCKS", "PLUS AND MATERNITY", "ELECTRONICS",
    "GIRLS WEAR, 4-6X  AND 7-14", "BRAS & SHAPEWEAR", "LIQUOR,WINE,BEER",
    "SLEEPWEAR/FOUNDATIONS", "CAMERAS AND SUPPLIES", "SPORTING GOODS",
    "PLAYERS AND ELECTRONICS", "PHARMACY RX", "MENSWEAR", "OPTICAL - FRAMES",
    "SWIMWEAR/OUTERWEAR", "OTHER DEPARTMENTS", "MEDIA AND GAMING", "FURNITURE",
    "OPTICAL - LENSES", "SEASONAL", "LARGE HOUSEHOLD GOODS", "ONE HR PHOTO",
    "CONCEPT STORES", "HEALTH AND BEAUTY AIDS",
]

safe = lambda s: re.sub(r'[^0-9A-Za-z]+', '_', s).upper().strip('_')
DEPT_COLS = [safe(d) for d in DEPARTMENTS]

print(NUM_NODES)

## Run Benchmark

Iterates over all scale factors. For each SF: runs preprocessing + training `NUM_RUNS` times, and writes results to the result table.

In [ ]:
from pyspark.sql import Row

phases = ["preprocess", "train"]
all_rows = []

for SF in SCALE_FACTORS:
    DATABASE = f"TPCXAI_V2_SF{SF}"
    
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    print(f"\n{'='*60}")
    print(f"SF={SF}  |  {NUM_NODES} node(s)  |  {NUM_RUNS} run(s)")
    print(f"{'='*60}")

    for run in range(1, NUM_RUNS + 1):
        timings = {}
        print(f"\n--- SF={SF} Run {run}/{NUM_RUNS} ---")

        # ---- Preprocess ----
        start = timeit.default_timer()

        orders   = spark.table(f"{DATABASE}.ORDERS")
        lineitem = spark.table(f"{DATABASE}.LINEITEM")
        product  = spark.table(f"{DATABASE}.PRODUCT")

        raw = (
            lineitem
            .join(orders, lineitem["LI_ORDER_ID"] == orders["O_ORDER_ID"])
            .join(product, lineitem["LI_PRODUCT_ID"] == product["P_PRODUCT_ID"])
            .filter(F.col("O_ORDER_ID").isNotNull())
            .filter(F.col("TRIP_TYPE") != 14)
        )

        # weekday one-hot + per-department quantity sums
        agg_exprs = [
            F.sum("QUANTITY").alias("SCAN_COUNT"),
            F.sum(F.abs(F.col("QUANTITY"))).alias("SCAN_COUNT_ABS"),
            F.min("TRIP_TYPE").alias("TRIP_TYPE"),
        ]
        for day in WEEKDAYS:
            agg_exprs.append(
                F.max(F.when(F.upper(F.col("WEEKDAY")) == day, 1).otherwise(0)).alias(day)
            )
        for dept, col_name in zip(DEPARTMENTS, DEPT_COLS):
            agg_exprs.append(
                F.sum(F.when(F.col("DEPARTMENT") == dept, F.col("QUANTITY")).otherwise(0)).alias(col_name)
            )

        features_df = raw.groupBy("O_ORDER_ID").agg(*agg_exprs).drop("O_ORDER_ID").fillna(0)
        features_df = features_df.withColumn(
            "TRIP_TYPE",
            F.dense_rank().over(Window.orderBy("TRIP_TYPE")) - 1
        )
        features_df = features_df.cache()
        row_count = features_df.count()  # materialize cache
        num_classes = features_df.agg(F.max("TRIP_TYPE")).collect()[0][0] + 1
        input_cols = [c for c in features_df.columns if c != "TRIP_TYPE"]

        timings["preprocess"] = timeit.default_timer() - start
        print(f"  Preprocess: {timings['preprocess']:.1f}s  |  {row_count:,} orders  |  {num_classes} classes")

        # ---- Train ----
        start = timeit.default_timer()

        assembler = VectorAssembler(inputCols=input_cols, outputCol="features")
        assembled_df = assembler.transform(features_df)

        # Free Spark cache before XGBoost allocates native memory
        features_df.unpersist()

        xgb = SparkXGBClassifier(
            features_col="features",
            label_col="TRIP_TYPE",
            num_workers=NUM_WORKERS,
            use_gpu=False,
            num_class=num_classes,
            tree_method="hist",
            n_estimators=100,
        )
        model = xgb.fit(assembled_df)

        timings["train"] = timeit.default_timer() - start
        print(f"  Train: {timings['train']:.1f}s  |  {len(input_cols)} features")

        # Clear all caches so next run starts cold
        features_df.unpersist()
        spark.catalog.clearCache()

        # Record results
        total = sum(timings[p] for p in phases)
        print(f"  Total: {total:.1f}s")

        for phase in phases:
            all_rows.append(Row(
                TIMESTAMP=ts, SF=SF, NUM_NODES=NUM_NODES,
                RUN=run, PHASE=phase, ELAPSED_S=round(timings[phase], 2),
                DBR_VERSION=DBR_VERSION,
                CONFIG=CONFIG,
            ))

    # Write after each scale factor completes (in case later ones fail)
    spark.createDataFrame(all_rows).write.mode("append").saveAsTable(RESULT_TABLE)
    print(f"\nResults for SF={SF} written to {RESULT_TABLE}")
    all_rows = []

print(f"\n{'='*60}")
print("All benchmarks complete!")
print(f"{'='*60}")